[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/17_dropout_interview.ipynb)

# 🟢 Solution: Implement Dropout（面试版）

## 📋 题目描述

**难度：** Easy

实现 Dropout（nn.Module）。

Dropout 在训练时以概率 p 随机将元素置零，并将存活元素缩放 `1/(1-p)` 以保持期望值不变。推理时为恒等映射。

**签名:** `MyDropout(p=0.5)`（nn.Module）

**前向传播:** `forward(x) -> Tensor`
- `x` — 任意形状的输入张量

**返回:** 应用 dropout 后的张量（训练）或原始输入（推理）

**约束:**
- 训练模式：以概率 p 置零，缩放 `1/(1-p)`
- 推理模式：返回原始输入

**提示：** 训练：`mask = (rand_like(x) > p).float()` → `x * mask / (1-p)`
推理：直接返回 `x`


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ INTERVIEW

class MyDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        # ---- Step 1: 保存丢弃概率 ----
        # p = 0 表示不丢弃，p = 1 表示全部丢弃（无意义）
        # 常见值：0.1 ~ 0.5
        self.p = p

    def forward(self, x):
        # ---- Step 2: 推理模式 ----
        # self.training 由 model.train() / model.eval() 控制
        # 推理时不做随机操作，保证输出确定性
        if not self.training or self.p == 0:
            return x

        # ---- Step 3: 生成随机掩码 ----
        # rand_like(x)：与 x 同 shape 的均匀分布 [0, 1)
        # > p：保留概率 = 1 - p
        mask = (torch.rand_like(x) > self.p).float()

        # ---- Step 4: inverted dropout 缩放 ----
        # 核心公式：output = x * mask / (1 - p)
        # 为什么叫"inverted"？
        #   早期 dropout 在推理时除以 (1-p)，训练时不缩放
        #   inverted 版本反过来：训练时除以 (1-p)，推理时不变
        #   inverted 版本更好：推理时零开销，不需要额外计算
        return x * mask / (1 - self.p)

# 面试常见追问：
# Q: Dropout 为什么能防过拟合？
# A: 1) 类似集成学习——每次前向传播随机丢不同子网络，相当于训练了指数级多个子模型
#    2) 迫使神经元不依赖特定的其他神经元，学到更鲁棒的特征
# Q: 为什么推理时不用 Dropout？
# A: 推理需要确定性输出。训练时已经用 (1-p) 缩放补偿过了，推理时直接用全网络即可。


In [ ]:
drop = MyDropout(p=0.5)
x = torch.ones(10)
drop.train()
print('Train:', drop(x))
drop.eval()
print('Eval: ', drop(x))


In [ ]:
from torch_judge import check
check('dropout')


## 📝 核心思路总结

1. **训练 vs 推理**：训练时随机置零 + 缩放，推理时恒等映射
2. **掩码生成**：`rand_like(x) > p` 生成伯努利掩码，保留概率 = 1-p
3. **Inverted Dropout**：训练时除以 (1-p) 保持期望不变，推理时零开销
4. **防过拟合原理**：类似集成学习 + 强制特征冗余
